<a href="https://colab.research.google.com/github/satyam-1605/RAG-from-basic-to-advance/blob/main/Hybrid_Search_in_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
doc_path="/content/Retrieval-Augmented-Generation-for-NLP.pdf"

In [ ]:
!pip install -q pypdf

In [ ]:
!pip install -q langchain_community

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:

loader=PyPDFLoader(doc_path)

In [ ]:
docs=loader.load()

In [ ]:

!pip install -qU langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=30)

In [ ]:
chunks = splitter.split_documents(docs)

In [ ]:
chunks

In [ ]:
!pip install -q langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:

from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HUGGINGFACE_TOKEN")

In [ ]:

HF_TOKEN = os.getenv("HF_TOKEN")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

In [ ]:
!pip install -qU "langchain-chroma"

In [ ]:
### Fixing OpenTelemetry version mismatch
!pip install -qU opentelemetry-api opentelemetry-sdk

After updating the packages, please **restart the session** (Runtime -> Restart session) for the changes to take effect, then run your Chroma import cell again.

In [ ]:

from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
)

In [ ]:


vector_store=Chroma.from_documents(chunks,embeddings)


In [ ]:

vectorstore_retreiver = vector_store.as_retriever(search_kwargs={"k": 3})

In [ ]:
vectorstore_retreiver

In [ ]:
!pip install  -q rank_bm25

In [ ]:
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever

In [ ]:
keyword_retriever = BM25Retriever.from_documents(chunks)

In [ ]:
keyword_retriever.k =  3

In [ ]:
ensemble_retriever = EnsembleRetriever(retrievers=[vectorstore_retreiver,keyword_retriever],weights=[0.3, 0.7])

In [ ]:
model_name = "HuggingFaceH4/zephyr-7b-beta"

In [ ]:
!pip install -q bitsandbytes

In [ ]:
!pip install -q accelerate

In [ ]:
import torch
from transformers import ( AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline, )
from langchain_huggingface.llms import HuggingFacePipeline

In [ ]:
# function for loading 4-bit quantized model
def load_quantized_model(model_name: str):
    """
    model_name: Name or path of the model to be loaded.
    return: Loaded quantized model.
    """
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        quantization_config=bnb_config,
    )
    return model

In [ ]:
# initializing tokenizer
def initialize_tokenizer(model_name: str):
    """
    model_name: Name or path of the model for tokenizer initialization.
    return: Initialized tokenizer.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name, return_token_type_ids=False)
    tokenizer.bos_token_id = 1  # Set beginning of sentence token id
    return tokenizer

In [ ]:
tokenizer = initialize_tokenizer(model_name)

In [ ]:
model = load_quantized_model(model_name)

In [ ]:
pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    use_cache=True,
    device_map="auto",
    max_length=2048,
    do_sample=True,
    top_k=5,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

In [ ]:
llm = HuggingFacePipeline(pipeline=pipeline)

In [ ]:
from langchain_classic.chains import RetrievalQA

In [ ]:
normal_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=vectorstore_retreiver
)

In [ ]:
hybrid_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ensemble_retriever
)

In [ ]:
response1 = normal_chain.invoke("What is Abstractive Question Answering?")

In [ ]:
response1

In [ ]:
print(response1.get("result"))

In [ ]:
response2 = hybrid_chain.invoke("What is Abstractive Question Answering?")

In [ ]:
response2

In [ ]:
print(response2.get("result"))